In [10]:
def grep(pattern):
    print("start grep for", pattern)
    while True:
        s = yield
        if pattern in s:
            print("found!", s)
        else:
            print("no %s in %s" % (pattern, s))

In [11]:
gr = grep("python")

In [3]:
next(gr)

start grep for python


In [4]:
gr.send("qlkdjlqjd")

no python in qlkdjlqjd


In [5]:
gr.send("python 3")

found! python 3


In [7]:
next(gr)

TypeError: argument of type 'NoneType' is not a container or iterable

In [9]:
gr.send("dwdw")

StopIteration: 

In [12]:
gr.throw(ValueError("wrong"))

ValueError: wrong

In [13]:
gr = grep("python")

In [14]:
next(gr)

start grep for python


In [15]:
gr.throw(ValueError("wrong"))

ValueError: wrong

In [17]:
def stepped_range(start, stop, step):
    print(f"start stepped_range {locals()=}")

    while start < stop:
        step_cand = yield start
        if isinstance(step_cand, int):
            step = step_cand

        start += step


stepped = stepped_range(-5, 10, 1)
assert next(stepped) == -5
assert next(stepped) == -4
assert stepped.send(4) == 0
assert next(stepped) == 4
assert stepped.send(-3) == 1
assert next(stepped) == -2
assert stepped.send(5) == 3
assert stepped.send(5) == 8
print("ok")

start stepped_range locals()={'start': -5, 'stop': 10, 'step': 1}
ok


In [18]:
next(stepped)

StopIteration: 

In [19]:
lst = [1, 2, 3]

In [20]:
hasattr(lst, "__iter__")

True

In [21]:
hasattr(lst, "__next__")

False

In [22]:
it1 = iter(lst)

In [23]:
it1

In [24]:
next(it1)

1

In [25]:
it2 = iter(it1)

In [26]:
next(it2)

2

In [27]:
it1 is it2

True

In [28]:
it3 = iter(lst)

In [29]:
it1 is it3

False

In [30]:
next(it3)

1

In [31]:
class SpecialIterator:
    def __init__(self, limit):
        self.limit = limit
        self.counter = 0

    def __iter__(self):
        print("iter")
        return self

    def __next__(self):
        print("next")
        if self.counter < self.limit:
            self.counter += 1
            return self.counter
        else:
            raise StopIteration

In [33]:
it = SpecialIterator(5)

In [34]:
type(it)

__main__.SpecialIterator

In [35]:
next(it)

next


1

In [36]:
next(it)

next


2

In [37]:
it2 = iter(it)

iter


In [38]:
it2 is it

True

In [39]:
next(it2)

next


3

In [40]:
next(it2)

next


4

In [41]:
next(it2)

next


5

In [42]:
next(it2)

next


StopIteration: 

In [43]:
for i in SpecialIterator(5):
    print(f"{i=}")

iter
next
i=1
next
i=2
next
i=3
next
i=4
next
i=5
next


In [46]:
list(zip([1, 2], [3, 4, 5]))

[(1, 3), (2, 4)]

In [47]:
it = iter(range(10))

result = list(
    zip(
        it,
        map(lambda x: x ** 2, it),
    )
)
print(result)

[(0, 1), (2, 9), (4, 25), (6, 49), (8, 81)]


In [48]:
import asyncio
import time
import aiohttp

In [54]:
async def say_after(delay, what):
    await asyncio.sleep(delay)
    print(what)


async def main():
    t1 = time.time()
    await say_after(1, "hello")
    print("after say 1")
    await say_after(2, "world")
    t2 = time.time()
    print("Total time:", t2 - t1)

In [55]:
await main()

hello
after say 1
world
Total time: 3.002549171447754


In [62]:
async def say_after(delay, what):
    print(f"start say_after {locals()=}")
    await asyncio.sleep(delay)
    print(what)


async def main():
    t1 = time.time()
    
    say1 = say_after(1, "hello")
    print("after say 1")
    await say_after(2, "world")
    await say1
    
    t2 = time.time()
    print("Total time:", t2 - t1)

await main()

after say 1
start say_after locals()={'delay': 2, 'what': 'world'}
world
start say_after locals()={'delay': 1, 'what': 'hello'}
hello
Total time: 3.0033979415893555


In [60]:
say1 = say_after(1, "hello")

/var/folders/j3/m947cq0j0fx8t0rrdzw5kmb00000gp/T/ipykernel_75725/3643469651.py:1: RuntimeWarning: coroutine 'say_after' was never awaited
  say1 = say_after(1, "hello")


In [63]:
await say1

start say_after locals()={'delay': 1, 'what': 'hello'}
hello


In [64]:
async def say_after(delay, what):
    print(f"start say_after {locals()=}")
    await asyncio.sleep(delay)
    print(what)


async def main():
    t1 = time.time()

    await asyncio.create_task(say_after(1, "hello"))
    print("after say 1")
    await asyncio.create_task(say_after(2, "world"))
    
    t2 = time.time()
    print("Total time:", t2 - t1)


await main()

start say_after locals()={'delay': 1, 'what': 'hello'}
hello
after say 1
start say_after locals()={'delay': 2, 'what': 'world'}
world
Total time: 3.004304885864258


In [66]:
async def say_after(delay, what):
    print(f"start say_after {locals()=}")
    await asyncio.sleep(delay)
    print(what)


async def main():
    t1 = time.time()

    task1 = asyncio.create_task(say_after(1, "hello"))
    print("after say 1")
    task2 = asyncio.create_task(say_after(2, "world"))

    await asyncio.gather(task1, task2)

    t2 = time.time()
    print("Total time:", t2 - t1)


await main()

after say 1
start say_after locals()={'delay': 1, 'what': 'hello'}
start say_after locals()={'delay': 2, 'what': 'world'}
hello
world
Total time: 2.0019030570983887


In [67]:
async def say_after(delay, what):
    print(f"start say_after {locals()=}")
    time.sleep(delay)
    print(what)


async def main():
    t1 = time.time()

    task1 = asyncio.create_task(say_after(1, "hello"))
    print("after say 1")
    task2 = asyncio.create_task(say_after(2, "world"))

    await asyncio.gather(task1, task2)

    t2 = time.time()
    print("Total time:", t2 - t1)


await main()

after say 1
start say_after locals()={'delay': 1, 'what': 'hello'}
hello
start say_after locals()={'delay': 2, 'what': 'world'}
world
Total time: 3.0081841945648193


In [68]:
async def say_after(delay, what):
    print(f"start say_after {locals()=}")
    await asyncio.to_thread(time.sleep, delay)
    print(what)
    return f"finished {what}"


async def main():
    t1 = time.time()

    task1 = asyncio.create_task(say_after(1, "hello"))
    print("after say 1")
    task2 = asyncio.create_task(say_after(2, "world"))
    #res = await task1

    res = await asyncio.gather(task1, task2)
    print("RESULTS:", res)

    t2 = time.time()
    print("Total time:", t2 - t1)


await main()

after say 1
start say_after locals()={'delay': 1, 'what': 'hello'}
start say_after locals()={'delay': 2, 'what': 'world'}
hello
world
RESULTS: ['finished hello', 'finished world']
Total time: 2.006664991378784


In [69]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"

In [95]:
async def fetch_url(url):
    async with aiohttp.ClientSession() as session:
        async with session.get(URL) as resp:
            return await resp.text()

In [93]:
status = await fetch_url(URL)

In [94]:
status

200

In [99]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"


async def fetch_url(url):
    async with aiohttp.ClientSession() as session:
        async with session.get(URL) as resp:
            return resp.status


async def fetch_batch_urls(urls):
    for url in urls:
        status = await fetch_url(url) 


async def run():
    t1 = time.time()
    urls = [URL] * 20

    res = await fetch_batch_urls(urls)
    t2 = time.time()
    print("total time", t2 - t1)


await run()

total time 5.494758367538452


In [105]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"


async def fetch_url(url, i):
    print("start fetch", i)
    async with aiohttp.ClientSession() as session:
        async with session.get(URL) as resp:
            await resp.text()
    print("finish fetch", i)


async def fetch_batch_urls(urls):
    tasks = []
    for i, url in enumerate(urls):
        tasks.append(fetch_url(url, i))

    await asyncio.gather(*tasks)


async def run():
    t1 = time.time()
    urls = [URL] * 10

    res = await fetch_batch_urls(urls)
    t2 = time.time()
    print("total time", t2 - t1)


await run()

start fetch 0
start fetch 1
start fetch 2
start fetch 3
start fetch 4
start fetch 5
start fetch 6
start fetch 7
start fetch 8
start fetch 9
finish fetch 5
finish fetch 3
finish fetch 2
finish fetch 7
finish fetch 4
finish fetch 9
finish fetch 1
finish fetch 6
finish fetch 8
finish fetch 0
total time 0.9887099266052246


In [108]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
WORKERS = 5


async def fetch_url(url):
    async with aiohttp.ClientSession() as session:
        async with session.get(URL) as resp:
            await resp.text()


async def worker(que, worker_name):
    print(f"start worker", worker_name)
    cnt = 0

    while True:
        url = await que.get()
        if url is None:
            await que.put(url)
            break

        res = await fetch_url(url)
        cnt += 1

    print(f"finish worker", worker_name, cnt)


async def run():
    t1 = time.time()
    urls = [URL] * 20
    
    que = asyncio.Queue()
    for url in urls:
        await que.put(url)
    await que.put(None)

    workers = [
        asyncio.create_task(worker(que, f"worker_{i}"))
        for i in range(WORKERS)
    ]
    await asyncio.gather(*workers)

    t2 = time.time()
    print("total time", t2 - t1)


await run()

start worker worker_0
start worker worker_1
start worker worker_2
start worker worker_3
start worker worker_4
finish worker worker_4 3
finish worker worker_1 4
finish worker worker_0 4
finish worker worker_3 5
finish worker worker_2 4
total time 2.0497190952301025
